# Lab 5: Path Planning — A* and RRT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatayer8891-boop/zuj-robotics/blob/main/labs/lab-05-path-planning/notebook.ipynb)

---

**Course:** Introduction to Robotics for AI and Data Science (0135343)  
**Instructor:** Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/)  
**University:** Al-Zaytoonah University of Jordan

---

## Overview

Implement A* search and Rapidly-exploring Random Trees (RRT) for collision-free path planning.

> **80/20 Principle:** Focus on understanding the core algorithm in each activity. The implementation details will follow naturally once you grasp the concept.


In [ ]:
# ============================================================
# COLAB ENVIRONMENT SETUP
# ============================================================
# This cell installs the required packages for running this lab
# in Google Colab. If you're running locally, you can skip this.
# ============================================================

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Setting up Colab environment...")
    !pip install pybullet numpy matplotlib opencv-python-headless -q
    print("✅ All packages installed!")
    print("⚠️  Note: PyBullet runs in HEADLESS mode on Colab (no 3D GUI).")
    print("   You'll see matplotlib plots instead of the live 3D window.")
    print("   For the full 3D experience, run locally with: conda activate robotics_YourName")
else:
    print("✅ Running locally — full GUI mode available.")


---

## Activity 1: Activity1 Astar

**File:** `activity1_astar.py`

Run the code below to complete this activity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import heapq

# ==============================================================================
# Activity 1: A* Grid Search (The Workhorse of 2D Planning)
# ==============================================================================
# Goal: Implement the A* algorithm to find the shortest path on a 2D grid.
# The 80/20 Principle: Mastering A* unlocks 80% of practical grid navigation.
#
# KEY CONCEPTS (from lecture):
#   C-space  : All possible robot states. Here, each grid cell is one state.
#   C_free   : Cells with value 0 — the robot can safely occupy these.
#   C_obs    : Cells with value 1 — the robot must never enter these.
#   f(n) = g(n) + h(n):
#       g(n) — exact cost from start to node n
#       h(n) — heuristic estimate from n to goal (must be ADMISSIBLE: never
#               overestimates, so A* stays optimal)
# ==============================================================================

# ------------------------------------------------------------------------------
# PART A — Heuristic
# ------------------------------------------------------------------------------

def heuristic(a, b):
    """
    Manhattan distance between two grid cells.

    Why Manhattan? Our robot moves in 4 directions (up/down/left/right),
    so the true minimum cost is |Δrow| + |Δcol| — never over-estimated.
    An admissible heuristic guarantees A* finds the OPTIMAL path.

    Args:
        a (tuple): (row, col) of node a
        b (tuple): (row, col) of node b

    Returns:
        int: Manhattan distance
    """
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


# ------------------------------------------------------------------------------
# PART B — Neighbour Expansion
# ------------------------------------------------------------------------------

def get_neighbors(node, grid):
    """
    Return valid 4-connected neighbours of 'node' that lie in C_free.

    Args:
        node  (tuple): current (row, col)
        grid  (ndarray): 2-D occupancy grid (0 = free, 1 = obstacle)

    Returns:
        list of (row, col) tuples
    """
    neighbors = []
    directions = [(0, 1), (0, -1), (1, 0), (-1, 0)]   # Right, Left, Down, Up

    for dr, dc in directions:
        nb = (node[0] + dr, node[1] + dc)

        # Bounds check
        if 0 <= nb[0] < grid.shape[0] and 0 <= nb[1] < grid.shape[1]:
            # C_free check: 0 = free space
            if grid[nb[0], nb[1]] == 0:
                neighbors.append(nb)

    return neighbors


# ------------------------------------------------------------------------------
# PART C — Core A* Algorithm
# ------------------------------------------------------------------------------

def a_star_search(grid, start, goal):
    """
    A* search on a 2-D occupancy grid.

    Algorithm outline:
      1. Initialise the OPEN SET (priority queue) with the start node.
      2. Pop the node with the lowest f = g + h.
      3. If it is the goal, reconstruct and return the path.
      4. Otherwise expand its neighbours, updating g and f scores.
      5. Repeat until the goal is found or the open set is empty (no path).

    Args:
        grid  (ndarray): 2-D occupancy grid (0 = free, 1 = obstacle)
        start (tuple) : (row, col) start position
        goal  (tuple) : (row, col) goal position

    Returns:
        path     (list of tuples | None): optimal path from start→goal, or None
        explored (set)                  : all nodes popped from the open set
    """
    # --- Open set: min-heap of (f_score, node) ---
    open_set = []
    heapq.heappush(open_set, (0, start))

    came_from = {}          # For path reconstruction
    g_score   = {start: 0} # Cost from start to each node
    explored  = set()       # Nodes already processed

    while open_set:
        # Always expand the node with the LOWEST f = g + h
        current_f, current = heapq.heappop(open_set)

        # Skip if already processed (a stale entry from a better path update)
        if current in explored:
            continue

        explored.add(current)

        # ---- Goal test ----
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            path.reverse()
            return path, explored

        # ---- Expand neighbours ----
        for neighbor in get_neighbors(current, grid):
            tentative_g = g_score[current] + 1   # uniform edge cost = 1

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor]   = tentative_g
                f_score             = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_score, neighbor))

    return None, explored   # No path exists


# ------------------------------------------------------------------------------
# PART D — Visualization
# ------------------------------------------------------------------------------

def visualize_grid(grid, start, goal, path=None, explored=None, title="A* Path Planning"):
    """
    Render the occupancy grid with explored nodes, final path, start, and goal.
    Also prints a statistics panel that mirrors the lecture slide metrics.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                             gridspec_kw={'width_ratios': [3, 1]})
    ax_grid, ax_stats = axes

    # ---- Grid ----
    ax_grid.imshow(grid, cmap='Greys', origin='upper', vmin=0, vmax=1)

    total_cells = grid.shape[0] * grid.shape[1]
    n_explored  = len(explored) if explored else 0
    path_len    = len(path)     if path     else 0

    # Explored nodes
    if explored:
        ex = [n[1] for n in explored]
        ey = [n[0] for n in explored]
        ax_grid.scatter(ex, ey, c='#AED6F1', marker='s', s=60,
                        alpha=0.6, label=f'Explored ({n_explored})')

    # Path
    if path:
        px = [n[1] for n in path]
        py = [n[0] for n in path]
        ax_grid.plot(px, py, c='#E74C3C', linewidth=3, label=f'Path ({path_len} steps)')

    # Start / Goal markers
    ax_grid.scatter(start[1], start[0], c='#2ECC71', marker='o',
                    s=200, zorder=5, label='Start')
    ax_grid.scatter(goal[1],  goal[0],  c='#3498DB', marker='*',
                    s=300, zorder=5, label='Goal')

    ax_grid.set_title(title, fontsize=13, fontweight='bold')
    ax_grid.legend(loc='upper left', fontsize=9)
    ax_grid.grid(True, color='gray', linestyle='-', linewidth=0.4, alpha=0.3)
    ax_grid.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax_grid.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax_grid.set_xticks([])
    ax_grid.set_yticks([])

    # C-space annotation
    ax_grid.text(0.01, 0.01,
                 "■ C_obs (obstacle)   □ C_free (navigable)",
                 transform=ax_grid.transAxes, fontsize=7,
                 color='gray', va='bottom')

    # ---- Stats panel ----
    ax_stats.axis('off')
    efficiency = (1 - n_explored / total_cells) * 100 if total_cells else 0

    stats = [
        ("PATH LENGTH",    f"{path_len} cells"   if path_len else "No path"),
        ("NODES EXPLORED", f"{n_explored}"),
        ("TOTAL CELLS",    f"{total_cells}"),
        ("EFFICIENCY",     f"{efficiency:.1f}%\n(cells skipped)"),
        ("HEURISTIC",      "Manhattan\n(admissible ✓)"),
        ("ALGORITHM",      "A* Search\nf(n) = g(n) + h(n)"),
    ]

    y = 0.97
    for label, value in stats:
        ax_stats.text(0.05, y, label, transform=ax_stats.transAxes,
                      fontsize=8, fontweight='bold', color='#555555', va='top')
        y -= 0.055
        ax_stats.text(0.05, y, value, transform=ax_stats.transAxes,
                      fontsize=11, color='#1A252F', va='top')
        y -= 0.10

    ax_stats.set_title("Statistics", fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig("activity1_output.png", dpi=120, bbox_inches='tight')
    print("[VIZ] Saved → activity1_output.png")
    plt.show()


# ------------------------------------------------------------------------------
# MAIN — Demo with warehouse-style obstacle layout
# ------------------------------------------------------------------------------

if __name__ == "__main__":
    # --- Build a 20×20 occupancy grid ---
    grid = np.zeros((20, 20))

    # Warehouse shelving units (C_obs)
    grid[5:15, 5]  = 1
    grid[5,  5:15] = 1
    grid[15, 10:18] = 1
    grid[10:18, 15] = 1

    start_node = (2,  2)
    goal_node  = (18, 18)

    print("=" * 55)
    print("  Activity 1 — A* Path Planning")
    print("=" * 55)
    print(f"  Grid size : {grid.shape[0]}×{grid.shape[1]} = {grid.size} cells")
    print(f"  C_obs     : {int(grid.sum())} obstacle cells")
    print(f"  Start     : {start_node}")
    print(f"  Goal      : {goal_node}")
    print("-" * 55)
    print("  Running A* …")

    path, explored = a_star_search(grid, start_node, goal_node)

    if path:
        efficiency = (1 - len(explored) / grid.size) * 100
        print(f"  ✅ Path found  : {len(path)} steps")
        print(f"  🔍 Explored    : {len(explored)} / {grid.size} nodes")
        print(f"  ⚡ Efficiency  : {efficiency:.1f}% of grid skipped")
        print("  (Compare: Dijkstra would explore ~60-80% of all cells)")
    else:
        print("  ❌ No path found.")

    print("=" * 55)
    visualize_grid(grid, start_node, goal_node, path, explored)


---

## Activity 2: Activity2 Rrt

**File:** `activity2_rrt.py`

Run the code below to complete this activity.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import random
import math
from itertools import product

# ==============================================================================
# Activity 2: RRT — Why Dimensionality Matters
# ==============================================================================
# Goal: Understand WHY RRT is necessary for high-dimensional spaces and see it
#       working across three progressively complex planning scenarios.
#
# The 80/20 Principle (Lecture Slide 4 — comparison table):
#   A* on grids    : grid memory scales as O(n^d) — a 10×10 grid in 6D
#                    = 10^6 = 1,000,000 cells. A 100×100 grid in 6D = 10^12.
#                    IMPOSSIBLE for robot arms.
#   RRT            : memory scales as O(iterations), not O(space size).
#                    Works in ANY number of dimensions with the same algorithm.
#
# Three demonstrations in this file:
#   Panel 1 (2-D)   — RRT in a flat corridor maze (baseline, like Dijkstra's)
#   Panel 2 (3-D)   — RRT navigating a 3-D volumetric space with spherical
#                     obstacles (impossible to grid-search efficiently)
#   Panel 3 (6-DOF) — RRT in the joint-space of a 6-degree-of-freedom robot
#                     arm. The C-space has 6 dimensions — grids are completely
#                     infeasible. RRT finds a path in joint-space that keeps
#                     the physical arm clear of a workspace obstacle.
#
# No coding required — run this file, observe the four-panel output, then
# answer the REFLECTION QUESTIONS at the bottom.
# ==============================================================================

SEED = 42


# ==============================================================================
# GENERIC RRT ENGINE  (works in N dimensions)
# ==============================================================================

class RRTNode:
    """A node in the RRT tree — holds an N-dimensional configuration."""
    def __init__(self, config):
        self.config = np.array(config, dtype=float)
        self.parent = None


def rrt_nd(start, goal, collision_fn, bounds,
           step_size=0.3, goal_bias=0.10, max_iter=2000):
    """
    Generic N-dimensional RRT planner.

    The exact same algorithm that works in 2-D also works in 3-D, 6-D, or
    any higher dimension — only the collision_fn and step_size change.

    Args:
        start        : array-like, start configuration
        goal         : array-like, goal  configuration
        collision_fn : callable(config) -> bool  (True = collision)
        bounds       : list of (min, max) per dimension
        step_size    : distance to extend toward random sample each iteration
        goal_bias    : probability of sampling the goal directly
        max_iter     : maximum tree-building iterations

    Returns:
        path  : list of np.arrays from start to goal (or None)
        nodes : all nodes added to the tree
    """
    random.seed(SEED)
    np.random.seed(SEED)

    dim   = len(start)
    root  = RRTNode(start)
    nodes = [root]

    def sample():
        if random.random() < goal_bias:
            return np.array(goal, dtype=float)
        return np.array([random.uniform(b[0], b[1]) for b in bounds])

    def nearest(q):
        dists = [np.linalg.norm(n.config - q) for n in nodes]
        return nodes[np.argmin(dists)]

    def steer(q_near, q_rand):
        direction = q_rand - q_near.config
        dist      = np.linalg.norm(direction)
        if dist < 1e-9:
            return q_near.config.copy()
        return q_near.config + (direction / dist) * min(step_size, dist)

    for _ in range(max_iter):
        q_rand   = sample()
        q_near   = nearest(q_rand)
        q_new_c  = steer(q_near, q_rand)

        if not collision_fn(q_new_c):
            new_node        = RRTNode(q_new_c)
            new_node.parent = q_near
            nodes.append(new_node)

            # Goal reached?
            if np.linalg.norm(new_node.config - np.array(goal)) < step_size:
                # Append goal node
                goal_node        = RRTNode(goal)
                goal_node.parent = new_node
                nodes.append(goal_node)

                # Reconstruct path
                path = []
                cur  = goal_node
                while cur is not None:
                    path.append(cur.config)
                    cur = cur.parent
                path.reverse()
                return path, nodes

    return None, nodes


# ==============================================================================
# DEMO 1 — 2-D Maze  (baseline)
# ==============================================================================

def build_2d_maze():
    """
    Three-room maze with two narrow doorways.
    Gap 1: x=3.0-3.5, y < 3.5  |  Gap 2: x=6.0-6.5, y > 6.0
    Verified path: (0.5,0.5) -> bottom gap -> middle -> top gap -> (9.5,9.5)
    """
    walls = [
        (3.0, 3.5, 3.5, 10.0),   # divider 1 -- gap at bottom (y < 3.5)
        (6.0, 6.5, 0.0,  6.0),   # divider 2 -- gap at top   (y > 6.0)
        (0.5, 2.5, 5.5,  6.0),   # shelf in room 1
        (7.5, 9.5, 3.5,  4.0),   # shelf in room 3
    ]

    def collision(q):
        x, y = q[0], q[1]
        if not (0 <= x <= 10 and 0 <= y <= 10):
            return True
        for (xm, xM, ym, yM) in walls:
            if xm <= x <= xM and ym <= y <= yM:
                return True
        return False

    return collision, walls


def plot_2d_rrt(ax, path, nodes, walls, start, goal):
    """Render the 2-D RRT tree and path."""
    ax.set_facecolor('#F8F9FA')
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', linewidth=0.35, alpha=0.4)

    # Walls
    for (xm, xM, ym, yM) in walls:
        ax.add_patch(plt.Rectangle((xm, ym), xM-xm, yM-ym,
                                    facecolor='#2C3E50', alpha=0.85))

    # Tree edges
    for node in nodes:
        if node.parent:
            ax.plot([node.parent.config[0], node.config[0]],
                    [node.parent.config[1], node.config[1]],
                    color='#82E0AA', linewidth=0.7, alpha=0.6)

    # Path
    if path:
        px = [p[0] for p in path]
        py = [p[1] for p in path]
        ax.plot(px, py, '-', color='#E74C3C', linewidth=2.5, zorder=5)

    ax.scatter(*start, c='#2ECC71', s=180, zorder=6, edgecolors='#1A8A45', lw=1.5)
    ax.scatter(*goal,  c='#3498DB', s=250, marker='*', zorder=6,
               edgecolors='#1A5276', lw=1.5)

    stats = (f"Dim: 2   |   Tree: {len(nodes)} nodes\n"
             f"Path: {len(path) if path else 'NONE'} pts   |   "
             f"Grid equiv: 100×100 = 10,000 cells")
    ax.text(0.02, 0.02, stats, transform=ax.transAxes, fontsize=7.5,
            va='bottom', color='#2C3E50',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.set_title("DEMO 1 — 2-D Continuous Space\n(Baseline)", fontsize=10,
                 fontweight='bold')
    ax.set_xlabel("x"); ax.set_ylabel("y")


# ==============================================================================
# DEMO 2 — 3-D Volumetric Space
# ==============================================================================

def build_3d_scene():
    """
    A 3-D space with spherical and cylindrical obstacles.
    A* would need a 3-D voxel grid. At 0.1 m resolution in 10m³ = 10^6 cells.
    """
    spheres   = [(3.0, 3.0, 5.0, 1.5),   # (cx, cy, cz, r)
                 (7.0, 5.0, 3.0, 1.2),
                 (5.0, 8.0, 7.0, 1.0)]
    cylinders = [(2.5, 7.5, 1.8)]         # (cx, cy, r) — infinite in z

    def collision(q):
        x, y, z = q
        if not (0 <= x <= 10 and 0 <= y <= 10 and 0 <= z <= 10):
            return True
        for cx, cy, cz, r in spheres:
            if (x-cx)**2 + (y-cy)**2 + (z-cz)**2 < r**2:
                return True
        for cx, cy, r in cylinders:
            if (x-cx)**2 + (y-cy)**2 < r**2:
                return True
        return False

    return collision, spheres, cylinders


def plot_3d_rrt(ax, path, nodes, spheres, cylinders, start, goal):
    """Render the 3-D RRT tree with volumetric obstacles."""
    ax.set_facecolor('#F8F9FA')
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_zlim(0, 10)
    ax.set_xlabel("x", fontsize=8); ax.set_ylabel("y", fontsize=8)
    ax.set_zlabel("z", fontsize=8)

    # Draw tree edges (thin, many)
    for node in nodes:
        if node.parent:
            ax.plot([node.parent.config[0], node.config[0]],
                    [node.parent.config[1], node.config[1]],
                    [node.parent.config[2], node.config[2]],
                    color='#82E0AA', linewidth=0.5, alpha=0.4)

    # Draw spherical obstacles as wireframe
    u = np.linspace(0, 2*np.pi, 20)
    v = np.linspace(0, np.pi, 15)
    for cx, cy, cz, r in spheres:
        xs = cx + r * np.outer(np.cos(u), np.sin(v))
        ys = cy + r * np.outer(np.sin(u), np.sin(v))
        zs = cz + r * np.outer(np.ones_like(u), np.cos(v))
        ax.plot_wireframe(xs, ys, zs, color='#2C3E50', alpha=0.25,
                          linewidth=0.5, rstride=3, cstride=3)

    # Draw cylindrical obstacles (as discs stacked)
    theta = np.linspace(0, 2*np.pi, 30)
    for cx, cy, r in cylinders:
        for zz in np.linspace(0, 10, 8):
            ax.plot(cx + r*np.cos(theta), cy + r*np.sin(theta),
                    np.full_like(theta, zz), color='#2C3E50', alpha=0.2, lw=0.8)

    # Path
    if path:
        px = [p[0] for p in path]
        py = [p[1] for p in path]
        pz = [p[2] for p in path]
        ax.plot(px, py, pz, '-', color='#E74C3C', linewidth=2.5, zorder=5)

    ax.scatter(*start, c='#2ECC71', s=120, zorder=6)
    ax.scatter(*goal,  c='#3498DB', s=160, marker='*', zorder=6)

    grid_equiv = "100³ = 1,000,000 voxels"
    stats = (f"Dim: 3   |   Tree: {len(nodes)} nodes\n"
             f"Grid equiv: {grid_equiv}")
    ax.set_title("DEMO 2 — 3-D Volumetric Space\n"
                 "(Grid: 1M voxels → RRT: same algorithm)", fontsize=10,
                 fontweight='bold')
    ax.text2D(0.02, 0.02, stats, transform=ax.transAxes, fontsize=7.5,
              bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))


# ==============================================================================
# DEMO 3 — 6-DOF Robot Arm in Joint Space
# ==============================================================================

LINK_LENGTHS = [1.0, 0.9, 0.8, 0.7, 0.5, 0.3]

def forward_kinematics_6dof(q):
    """
    Simplified planar 6-DOF arm forward kinematics.
    Each joint rotates in the XY plane, stacking cumulative angle.
    Returns the (x, y) positions of all joint origins + end-effector.
    """
    joints = [(0.0, 0.0)]
    cumulative_angle = 0.0
    x, y = 0.0, 0.0
    for i, (angle, length) in enumerate(zip(q, LINK_LENGTHS)):
        cumulative_angle += angle
        x += length * math.cos(cumulative_angle)
        y += length * math.sin(cumulative_angle)
        joints.append((x, y))
    return joints


def arm_workspace_collision(q, obstacles):
    """
    True if the arm (in config q) intersects any circular workspace obstacle.
    This maps from 6-D joint space → 2-D workspace → collision check.
    """
    joints = forward_kinematics_6dof(q)
    for i in range(len(joints) - 1):
        x1, y1 = joints[i]
        x2, y2 = joints[i+1]
        for cx, cy, r in obstacles:
            # Check segment–circle intersection
            dx, dy   = x2 - x1, y2 - y1
            fx, fy   = x1 - cx, y1 - cy
            a        = dx*dx + dy*dy
            b        = 2*(fx*dx + fy*dy)
            c        = fx*fx + fy*fy - r*r
            disc     = b*b - 4*a*c
            if disc >= 0:
                t1 = (-b - math.sqrt(disc)) / (2*a)
                t2 = (-b + math.sqrt(disc)) / (2*a)
                if 0 <= t1 <= 1 or 0 <= t2 <= 1:
                    return True
    return False


def draw_arm(ax, q, color='#7D3C98', alpha=1.0, lw=2.5, label=None):
    """Draw the 6-link arm for a given joint configuration."""
    joints = forward_kinematics_6dof(q)
    xs = [j[0] for j in joints]
    ys = [j[1] for j in joints]
    ax.plot(xs, ys, '-o', color=color, linewidth=lw, alpha=alpha,
            markersize=5, markerfacecolor=color, label=label)
    return joints


def plot_6dof_panel(ax_arm, ax_cspace, path, nodes,
                    obstacles, q_start, q_goal):
    """
    Two sub-axes for the 6-DOF demo:
      ax_arm    — physical workspace showing arm start, goal, and path samples
      ax_cspace — projection of the 6-D joint space tree onto the first 2 joints
    """
    # ── Workspace panel ───────────────────────────────────────────────────
    ax_arm.set_facecolor('#F0F0F0')
    ax_arm.set_xlim(-4.5, 4.5); ax_arm.set_ylim(-4.5, 4.5)
    ax_arm.set_aspect('equal')
    ax_arm.grid(True, linestyle='--', linewidth=0.35, alpha=0.4)
    ax_arm.axhline(0, color='gray', lw=0.5)
    ax_arm.axvline(0, color='gray', lw=0.5)

    # Obstacles in workspace
    for cx, cy, r in obstacles:
        ax_arm.add_patch(plt.Circle((cx, cy), r,
                                     facecolor='#E74C3C', alpha=0.45,
                                     edgecolor='#922B21', linewidth=1.5))
        ax_arm.text(cx, cy, 'Obstacle', ha='center', va='center',
                    fontsize=6.5, color='white', fontweight='bold')

    # Draw a selection of intermediate arm configs along the path
    if path and len(path) > 4:
        step = max(1, len(path) // 6)
        for i in range(0, len(path), step):
            draw_arm(ax_arm, path[i], color='#AED6F1', alpha=0.5, lw=1.0)

    # Start and goal arms on top
    draw_arm(ax_arm, q_start, color='#2ECC71', lw=2.5, label='Start config')
    draw_arm(ax_arm, q_goal,  color='#3498DB', lw=2.5, label='Goal config')

    # End-effector trajectory
    if path:
        ee = [forward_kinematics_6dof(q)[-1] for q in path]
        ax_arm.plot([p[0] for p in ee], [p[1] for p in ee],
                    '-', color='#E74C3C', lw=2.0, alpha=0.85, zorder=4,
                    label='End-effector path')

    ax_arm.scatter(0, 0, c='black', s=80, zorder=6)   # base
    ax_arm.legend(fontsize=7, loc='upper right')
    ax_arm.set_title("6-DOF Arm — Physical Workspace\n"
                     "Light blue = intermediate configurations", fontsize=10,
                     fontweight='bold')
    ax_arm.set_xlabel("x (m)"); ax_arm.set_ylabel("y (m)")

    # ── Joint-space projection panel (θ₁ vs θ₂) ─────────────────────────
    ax_cspace.set_facecolor('#F8F9FA')
    ax_cspace.set_xlim(-math.pi, math.pi)
    ax_cspace.set_ylim(-math.pi, math.pi)
    ax_cspace.grid(True, linestyle='--', linewidth=0.35, alpha=0.4)

    # Draw tree edges projected to (θ₁, θ₂)
    for node in nodes:
        if node.parent:
            ax_cspace.plot(
                [node.parent.config[0], node.config[0]],
                [node.parent.config[1], node.config[1]],
                color='#82E0AA', linewidth=0.7, alpha=0.55)

    # Path in joint space
    if path:
        ax_cspace.plot([q[0] for q in path], [q[1] for q in path],
                       '-', color='#E74C3C', linewidth=2.5, zorder=5)

    ax_cspace.scatter(q_start[0], q_start[1], c='#2ECC71', s=180, zorder=6,
                      edgecolors='#1A8A45', lw=1.5, label='Start')
    ax_cspace.scatter(q_goal[0],  q_goal[1],  c='#3498DB', s=250, marker='*',
                      zorder=6, edgecolors='#1A5276', lw=1.5, label='Goal')

    ax_cspace.set_xlabel("θ₁ — Joint 1 [rad]", fontsize=9)
    ax_cspace.set_ylabel("θ₂ — Joint 2 [rad]", fontsize=9)
    ax_cspace.legend(fontsize=7)

    grid_str = ("A* grid at 10°-resolution:\n"
                "36⁶ = 2,176,782,336 cells\n"
                "→ completely infeasible\n\n"
                f"RRT tree: {len(nodes)} nodes\n"
                f"Path: {len(path) if path else 'NONE'} configs\n"
                "→ works perfectly")
    ax_cspace.text(0.97, 0.97, grid_str, transform=ax_cspace.transAxes,
                   fontsize=8, va='top', ha='right',
                   bbox=dict(boxstyle='round', facecolor='#FEF9E7', alpha=0.9))
    ax_cspace.set_title("6-DOF C-Space: θ₁ vs θ₂ Projection\n"
                         "(Full space is 6-dimensional)", fontsize=10,
                         fontweight='bold')


# ==============================================================================
# DIMENSION SCALING COMPARISON PANEL
# ==============================================================================

def plot_scaling_panel(ax):
    """
    Log-scale plot comparing grid cell count vs RRT node count
    as dimensionality increases. Makes the curse of dimensionality visual.
    """
    dims        = np.arange(2, 9)
    resolution  = 20              # 20 divisions per dimension
    grid_cells  = resolution ** dims

    # RRT uses roughly the same number of nodes regardless of dimension
    # (empirically ~500-2000 for similar environments)
    rrt_nodes_lo = np.full_like(dims, 500,  dtype=float)
    rrt_nodes_hi = np.full_like(dims, 2000, dtype=float)

    ax.fill_between(dims, rrt_nodes_lo, rrt_nodes_hi,
                    alpha=0.25, color='#27AE60', label='RRT (typical range)')
    ax.plot(dims, rrt_nodes_lo, '--', color='#27AE60', lw=1.5)
    ax.plot(dims, rrt_nodes_hi, '--', color='#27AE60', lw=1.5)
    ax.plot(dims, grid_cells, '-o', color='#E74C3C', lw=2.5, ms=7,
            label=f'Grid (res={resolution} per axis)')

    # Annotate specific values
    for d, g in zip(dims, grid_cells):
        ax.annotate(f'{g:.0e}', (d, g), textcoords='offset points',
                    xytext=(5, 4), fontsize=7.5, color='#922B21')

    # Mark the three demos
    demo_dims  = [2, 3, 6]
    demo_names = ['Demo 1\n2-D maze', 'Demo 2\n3-D space', 'Demo 3\n6-DOF arm']
    demo_colors= ['#2ECC71', '#3498DB', '#8E44AD']
    for d, name, c in zip(demo_dims, demo_names, demo_colors):
        ax.axvline(d, color=c, linestyle=':', linewidth=1.5, alpha=0.7)
        ax.text(d, ax.get_ylim()[1] if ax.get_ylim()[1] > 1 else 1e10,
                name, ha='center', fontsize=7.5, color=c, va='top')

    ax.set_yscale('log')
    ax.set_xlabel("Number of Dimensions  (DOF)", fontsize=10)
    ax.set_ylabel("Number of Cells / Nodes (log scale)", fontsize=10)
    ax.set_title("Curse of Dimensionality:\nWhy Grids Fail and RRT Scales",
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.set_xticks(dims)
    ax.set_xticklabels([f'{d}-D' for d in dims])
    ax.grid(True, which='both', linestyle='--', alpha=0.4)

    # Annotation arrow
    ax.annotate("Grid grows\nexponentially",
                xy=(6, resolution**6), xytext=(5.3, resolution**6 * 5),
                fontsize=8, color='#922B21',
                arrowprops=dict(arrowstyle='->', color='#922B21'))
    ax.annotate("RRT stays\nconstant",
                xy=(6, 1000), xytext=(6.4, 300),
                fontsize=8, color='#27AE60',
                arrowprops=dict(arrowstyle='->', color='#27AE60'))


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    print("=" * 65)
    print("  Activity 2 — RRT: Why Dimensionality Matters")
    print("=" * 65)
    print()

    # ── Run all three RRT demos ───────────────────────────────────────────

    # Demo 1 — 2-D maze
    print("[DEMO 1] 2-D maze ...")
    col_2d, walls = build_2d_maze()
    path_2d, nodes_2d = rrt_nd(
        start=[0.5, 0.5], goal=[9.5, 9.5],
        collision_fn=col_2d,
        bounds=[(0, 10), (0, 10)],
        step_size=0.5, goal_bias=0.12, max_iter=3000)
    print(f"  Tree: {len(nodes_2d)} nodes | Path: "
          f"{len(path_2d) if path_2d else 'NOT FOUND'} pts")

    # Demo 2 — 3-D volumetric
    print("[DEMO 2] 3-D volumetric space ...")
    col_3d, spheres, cylinders = build_3d_scene()
    path_3d, nodes_3d = rrt_nd(
        start=[0.5, 0.5, 0.5], goal=[9.5, 9.5, 9.5],
        collision_fn=col_3d,
        bounds=[(0, 10)] * 3,
        step_size=0.7, goal_bias=0.12, max_iter=4000)
    print(f"  Tree: {len(nodes_3d)} nodes | Path: "
          f"{len(path_3d) if path_3d else 'NOT FOUND'} pts")

    # Demo 3 — 6-DOF robot arm
    print("[DEMO 3] 6-DOF robot arm in joint space ...")
    obstacles_ws = [(1.5, 1.0, 0.5), (-0.5, 2.0, 0.45)]

    q_start = [0.3,  0.2, -0.1,  0.15,  0.1, -0.05]
    q_goal  = [-0.8, 0.6,  0.5, -0.3,  -0.2,  0.4 ]

    bounds_6dof = [(-math.pi, math.pi)] * 6

    def col_6dof(q):
        return arm_workspace_collision(q, obstacles_ws)

    path_6d, nodes_6d = rrt_nd(
        start=q_start, goal=q_goal,
        collision_fn=col_6dof,
        bounds=bounds_6dof,
        step_size=0.25, goal_bias=0.10, max_iter=5000)
    print(f"  Tree: {len(nodes_6d)} nodes | Path: "
          f"{len(path_6d) if path_6d else 'NOT FOUND'} configs")

    # ── Build the figure ──────────────────────────────────────────────────
    print()
    print("[VIZ] Building multi-panel figure ...")

    fig = plt.figure(figsize=(20, 16))
    fig.patch.set_facecolor('#FAFAFA')

    # Layout: 2 rows × 3 cols, with bottom-left spanning 2 cols
    gs = gridspec.GridSpec(2, 3, figure=fig,
                            hspace=0.38, wspace=0.32,
                            left=0.06, right=0.97,
                            top=0.92, bottom=0.06)

    ax1     = fig.add_subplot(gs[0, 0])               # 2-D
    ax2     = fig.add_subplot(gs[0, 1], projection='3d')  # 3-D
    ax_scale= fig.add_subplot(gs[0, 2])               # scaling chart
    ax_arm  = fig.add_subplot(gs[1, 0])               # 6-DOF workspace
    ax_cs   = fig.add_subplot(gs[1, 1])               # 6-DOF c-space
    ax_ref  = fig.add_subplot(gs[1, 2])               # reflection questions

    # Panel 1
    plot_2d_rrt(ax1, path_2d, nodes_2d, walls,
                start=[0.5, 0.5], goal=[9.5, 9.5])

    # Panel 2
    plot_3d_rrt(ax2, path_3d, nodes_3d, spheres, cylinders,
                start=[0.5, 0.5, 0.5], goal=[9.5, 9.5, 9.5])

    # Panel 3 — Scaling chart
    plot_scaling_panel(ax_scale)

    # Panel 4+5 — 6-DOF arm
    plot_6dof_panel(ax_arm, ax_cs, path_6d, nodes_6d,
                    obstacles_ws, q_start, q_goal)

    # Panel 6 — Reflection questions
    ax_ref.axis('off')
    reflection = (
        "WHAT TO OBSERVE\n"
        "─────────────────────────────────────────\n\n"
        "Demo 1 (2-D):\n"
        "  The tree fans out through corridors.\n"
        "  Notice the jagged path — RRT is NOT\n"
        "  optimal. A* would find a shorter path.\n\n"
        "Demo 2 (3-D):\n"
        "  Same RRT algorithm, one extra dimension.\n"
        "  A* would need 1,000,000 voxels at\n"
        "  0.1 m resolution — RRT uses ~700 nodes.\n\n"
        "Demo 3 (6-DOF arm):\n"
        "  Planning happens in 6-D joint space.\n"
        "  A 10-degree grid would need 36⁶ ≈ 2.2B\n"
        "  cells. RRT needs ~800 nodes. The tree in\n"
        "  the C-space panel (θ₁ vs θ₂) is the 2-D\n"
        "  projection of a 6-D tree.\n\n"
        "─────────────────────────────────────────\n"
        "REFLECTION QUESTIONS\n"
        "─────────────────────────────────────────\n\n"
        "Q1. Path quality\n"
        "    RRT paths look jagged in all 3 demos.\n"
        "    Why? What algorithm variant would\n"
        "    produce a smoother path?\n\n"
        "Q2. Scaling\n"
        "    From the chart: at 6-D with resolution\n"
        "    20, how many grid cells are needed?\n"
        "    How does RRT's node count compare?\n\n"
        "Q3. Algorithm choice\n"
        "    A surgical robot has 7 joints (7-DOF).\n"
        "    Would you use A* or RRT? Justify using\n"
        "    the comparison table from Lecture Slide 4."
    )
    ax_ref.text(0.03, 0.97, reflection, transform=ax_ref.transAxes,
                fontsize=8.5, va='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='#FEF9E7', alpha=0.95))
    ax_ref.set_title("Observations & Reflection", fontsize=10,
                     fontweight='bold')

    # Super-title
    fig.suptitle(
        "Activity 2 — RRT: Rapidly-exploring Random Trees Across Dimensions\n"
        "2-D Maze  |  3-D Volumetric Space  |  6-DOF Robot Arm Joint Space",
        fontsize=13, fontweight='bold', y=0.97)

    plt.savefig("activity2_output.png", dpi=130, bbox_inches='tight')
    print("[VIZ] Saved --> activity2_output.png")

    # ── Console summary ───────────────────────────────────────────────────
    print()
    print("=" * 65)
    print("  RESULTS SUMMARY")
    print("=" * 65)
    print(f"  Demo 1 (2-D) : {len(nodes_2d):>5} tree nodes | "
          f"{len(path_2d) if path_2d else 'N/A':>4} path pts | "
          f"grid equiv: {20**2:>10,} cells")
    print(f"  Demo 2 (3-D) : {len(nodes_3d):>5} tree nodes | "
          f"{len(path_3d) if path_3d else 'N/A':>4} path pts | "
          f"grid equiv: {20**3:>10,} cells")
    print(f"  Demo 3 (6-D) : {len(nodes_6d):>5} tree nodes | "
          f"{len(path_6d) if path_6d else 'N/A':>4} path pts | "
          f"grid equiv: {20**6:>10,} cells")
    print()
    print("  Key insight: RRT node count stays in the hundreds regardless")
    print("  of dimension. Grid size grows exponentially (curse of")
    print("  dimensionality). At 6-D, a grid is ~64 million times larger.")
    print()
    print("REFLECTION QUESTIONS (answer in your lab report):")
    print("  Q1. Why are all three RRT paths jagged? What variant fixes this?")
    print("  Q2. From the chart: what is the grid size at 6-D, res=20?")
    print("  Q3. A 7-DOF surgical robot: A* or RRT? Justify your choice.")
    print("=" * 65)

    plt.show()


if __name__ == "__main__":
    main()


---

## Activity 3: Activity3 Warehouse

**File:** `activity3_warehouse.py`

Run the code below to complete this activity.


In [ ]:
import pybullet as p
import pybullet_data
import time
import math
import numpy as np
import sys
import os

# ==============================================================================
# Activity 3: PyBullet Warehouse — Path Planning Visualization
# ==============================================================================
# Goal: Use your A* implementation from Activity 1 to plan a collision-free
#       path through a 3-D warehouse, and visualize it in PyBullet.
#
# IMPORTANT — What this activity covers and what comes next:
#   THIS LAB   : Path PLANNING — computing the sequence of waypoints from
#                start to goal using A* on the warehouse occupancy grid.
#   NEXT LABS  : Path TRACKING & CONTROL — how to generate motor commands
#                that make the robot physically follow those waypoints.
#
# There is NO robot movement in this simulation. The Husky robot is placed
# at the start position and stays still. What you WILL see:
#   - The 3-D warehouse with shelving-unit obstacles
#   - The A* planned path drawn as a glowing line through the warehouse
#   - Coloured waypoint spheres along the path
#   - A matplotlib top-down summary plot saved to disk
#
# Architecture (Lecture Slide 9 — focus on Layer 1 only today):
#   LAYER 1 — Global Planner (A*)   <-- THIS ACTIVITY
#   Layer 2 — Local Planner         <-- Lab 6
#   Layer 3 — Controller            <-- Lab 6
#
# What changed from the previous version:
#   - Controller (Layer 2 & 3) removed entirely — not the topic of this lab
#   - Scenario B (local minima) removed — belongs in the control lab
#   - Focus is 100% on visualizing what A* produces in 3-D space
#   - Path is drawn as debug lines + waypoint spheres directly in PyBullet
#   - Robot is loaded purely for visual context — it does not move
# ==============================================================================

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from activity1_astar import a_star_search


# ==============================================================================
# GRID CONFIGURATION
# ==============================================================================

CELL_SIZE  = 1.0      # metres per grid cell
GRID_ROWS  = 12
GRID_COLS  = 12
WALL_HEIGHT = 1.2     # height of shelf obstacles in the 3-D scene [metres]


def cell_to_world(row, col):
    """Convert grid (row, col) to continuous world (x, y) at the cell centre."""
    return col * CELL_SIZE + CELL_SIZE / 2, row * CELL_SIZE + CELL_SIZE / 2


def build_warehouse_grid():
    """
    12x12 occupancy grid representing the warehouse shelving layout.
    0 = C_free (navigable aisle), 1 = C_obs (shelf unit).

    This grid is the ONLY input to A*. The planner has no knowledge of
    the 3-D geometry — it works purely on this 2-D abstraction.
    """
    grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=int)

    # Left shelf block
    grid[3:7, 2]   = 1
    grid[3,   2:6] = 1

    # Right shelf block
    grid[3:7, 8]   = 1
    grid[3,   7:9] = 1

    # Bottom aisle divider
    grid[8, 3:9]   = 1

    return grid


# ==============================================================================
# GLOBAL PLANNER  (A* from Activity 1)
# ==============================================================================

def run_global_planner(grid, start_cell, goal_cell):
    """
    Run A* on the occupancy grid and return world-coordinate waypoints.

    Args:
        grid       (ndarray) : 2-D occupancy grid (0 = free, 1 = obstacle)
        start_cell (tuple)   : (row, col) start in grid coordinates
        goal_cell  (tuple)   : (row, col) goal  in grid coordinates

    Returns:
        waypoints (list of [x, y]) : world-coordinate path points
        explored  (set)            : all cells A* examined
    """
    print("[A*] Planning path on warehouse occupancy grid ...")
    path, explored = a_star_search(grid, start_cell, goal_cell)

    if path is None:
        print("[A*] ERROR: No path found — check start/goal are in C_free.")
        return None, explored

    waypoints = [list(cell_to_world(r, c)) for (r, c) in path]

    print(f"[A*] Path found:  {len(waypoints)} waypoints")
    print(f"[A*] Explored:    {len(explored)} / {grid.size} cells "
          f"({100*len(explored)/grid.size:.1f}%)")
    print(f"[A*] Path length: {len(waypoints) - 1} steps")
    return waypoints, explored


# ==============================================================================
# PYBULLET SCENE BUILDER
# ==============================================================================

def setup_pybullet():
    """Connect to PyBullet GUI and configure the viewer."""
    client = p.connect(p.DIRECT if IN_COLAB else p.GUI)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)

    # Camera: top-down view showing the full warehouse
    p.resetDebugVisualizerCamera(
        cameraDistance    = 16,
        cameraYaw         = 30,
        cameraPitch       = -50,
        cameraTargetPosition = [6, 6, 0])

    # Clean GUI — hide overlay panels
    p.configureDebugVisualizer(p.COV_ENABLE_GUI,            0)
    p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS,        1)
    p.configureDebugVisualizer(p.COV_ENABLE_RGB_BUFFER_PREVIEW, 0)

    return client


def load_ground_and_walls():
    """Load ground plane and low boundary walls."""
    p.loadURDF("plane.urdf")

    # Thin boundary walls (visual only)
    wall_color  = [0.55, 0.55, 0.60, 1.0]
    world_w = GRID_COLS * CELL_SIZE
    world_h = GRID_ROWS * CELL_SIZE

    for pos, half in [
        ([world_w/2, -0.1,       0.3], [world_w/2, 0.05, 0.3]),
        ([world_w/2, world_h+0.1, 0.3], [world_w/2, 0.05, 0.3]),
        ([-0.1,       world_h/2, 0.3], [0.05, world_h/2, 0.3]),
        ([world_w+0.1, world_h/2, 0.3], [0.05, world_h/2, 0.3]),
    ]:
        col = p.createCollisionShape(p.GEOM_BOX, halfExtents=half)
        vis = p.createVisualShape(p.GEOM_BOX, halfExtents=half,
                                   rgbaColor=wall_color)
        p.createMultiBody(baseMass=0,
                          baseCollisionShapeIndex=col,
                          baseVisualShapeIndex=vis,
                          basePosition=pos)


def load_warehouse_obstacles(grid):
    """
    Instantiate 3-D shelf boxes for every C_obs cell in the grid.
    Shelves are tall brown boxes with a darker top cap.
    """
    shelf_body_color = [0.55, 0.38, 0.18, 1.0]   # wood brown
    shelf_top_color  = [0.35, 0.22, 0.08, 1.0]   # darker top

    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            if grid[r, c] == 1:
                x, y = cell_to_world(r, c)

                # Main shelf body
                col = p.createCollisionShape(
                    p.GEOM_BOX,
                    halfExtents=[CELL_SIZE/2 - 0.04,
                                 CELL_SIZE/2 - 0.04,
                                 WALL_HEIGHT / 2])
                vis = p.createVisualShape(
                    p.GEOM_BOX,
                    halfExtents=[CELL_SIZE/2 - 0.04,
                                 CELL_SIZE/2 - 0.04,
                                 WALL_HEIGHT / 2],
                    rgbaColor=shelf_body_color)
                p.createMultiBody(baseMass=0,
                                  baseCollisionShapeIndex=col,
                                  baseVisualShapeIndex=vis,
                                  basePosition=[x, y, WALL_HEIGHT / 2])

                # Top cap (darker stripe)
                vis_top = p.createVisualShape(
                    p.GEOM_BOX,
                    halfExtents=[CELL_SIZE/2 - 0.04,
                                 CELL_SIZE/2 - 0.04,
                                 0.05],
                    rgbaColor=shelf_top_color)
                p.createMultiBody(baseMass=0,
                                  baseCollisionShapeIndex=-1,
                                  baseVisualShapeIndex=vis_top,
                                  basePosition=[x, y, WALL_HEIGHT + 0.05])


def load_robot(start_cell):
    """
    Place the Husky robot at the start position.
    The robot does NOT move — it is here for visual context only.
    Path tracking will be covered in Lab 6.
    """
    x, y = cell_to_world(*start_cell)
    ori  = p.getQuaternionFromEuler([0, 0, 0])
    robot_id = p.loadURDF("husky/husky.urdf", [x, y, 0.15], ori)

    # Let physics settle for a moment
    for _ in range(80):
        p.stepSimulation()

    print(f"[ROBOT] Husky placed at world ({x:.1f}, {y:.1f})")
    print("[ROBOT] Note: robot is stationary — path tracking is Lab 6.")
    return robot_id


# ==============================================================================
# PATH VISUALIZATION IN PYBULLET
# ==============================================================================

def visualize_path_in_pybullet(waypoints, explored, grid):
    """
    Draw the A* result directly into the 3-D PyBullet scene:

      1. Explored cells  — flat semi-transparent blue tiles on the ground
      2. Path line       — bright glowing line connecting all waypoints
      3. Waypoint spheres— small coloured spheres at each waypoint
      4. Goal marker     — larger green sphere at the destination
    """
    path_height    = 0.25    # height to draw path line above ground
    sphere_radius  = 0.15
    goal_radius    = 0.30

    # ── 1. Explored cells (subtle floor tiles) ───────────────────────────
    explored_color = [0.40, 0.65, 0.95, 0.30]
    for (r, c) in explored:
        if grid[r, c] == 0:   # only draw on free cells
            x, y = cell_to_world(r, c)
            vis  = p.createVisualShape(
                p.GEOM_BOX,
                halfExtents=[CELL_SIZE/2 - 0.06,
                              CELL_SIZE/2 - 0.06,
                              0.01],
                rgbaColor=explored_color)
            p.createMultiBody(baseMass=0,
                               baseCollisionShapeIndex=-1,
                               baseVisualShapeIndex=vis,
                               basePosition=[x, y, 0.01])

    # ── 2. Path line (bright red debug lines) ────────────────────────────
    for i in range(len(waypoints) - 1):
        p.addUserDebugLine(
            lineFromXYZ = [waypoints[i][0],   waypoints[i][1],   path_height],
            lineToXYZ   = [waypoints[i+1][0], waypoints[i+1][1], path_height],
            lineColorRGB = [1.0, 0.15, 0.15],
            lineWidth    = 4)

    # ── 3. Waypoint spheres (gradient: cyan at start → orange at end) ────
    n = len(waypoints)
    for i, wp in enumerate(waypoints):
        t = i / max(n - 1, 1)   # 0.0 at start, 1.0 at goal
        r_c = t                  # red increases toward goal
        g_c = 1.0 - t * 0.7     # green fades
        b_c = 1.0 - t           # blue fades to zero

        vis = p.createVisualShape(
            p.GEOM_SPHERE,
            radius    = sphere_radius,
            rgbaColor = [r_c, g_c, b_c, 0.85])
        p.createMultiBody(baseMass=0,
                           baseCollisionShapeIndex=-1,
                           baseVisualShapeIndex=vis,
                           basePosition=[wp[0], wp[1], path_height])

    # ── 4. Goal marker (large green sphere) ──────────────────────────────
    goal_wp = waypoints[-1]
    vis_goal = p.createVisualShape(
        p.GEOM_SPHERE,
        radius    = goal_radius,
        rgbaColor = [0.10, 0.90, 0.25, 0.90])
    p.createMultiBody(baseMass=0,
                       baseCollisionShapeIndex=-1,
                       baseVisualShapeIndex=vis_goal,
                       basePosition=[goal_wp[0], goal_wp[1], path_height + 0.1])

    # ── 5. Text labels ────────────────────────────────────────────────────
    start_wp = waypoints[0]
    p.addUserDebugText("START",
                        [start_wp[0], start_wp[1], path_height + 0.5],
                        textColorRGB=[0.1, 0.9, 0.3],
                        textSize=1.2)
    p.addUserDebugText("GOAL",
                        [goal_wp[0], goal_wp[1], path_height + 0.6],
                        textColorRGB=[0.1, 0.9, 0.3],
                        textSize=1.2)
    p.addUserDebugText(
        f"A* path: {len(waypoints)} waypoints  |  "
        f"Explored: {len(explored)} cells",
        [GRID_COLS * CELL_SIZE / 2, -0.8, 0.3],
        textColorRGB=[0.9, 0.9, 0.2],
        textSize=0.9)


# ==============================================================================
# MATPLOTLIB SUMMARY PLOT
# ==============================================================================

def save_summary_plot(grid, waypoints, explored, start_cell, goal_cell):
    """
    Save a clean top-down 2-D summary plot of the A* result.
    Shows the occupancy grid, explored region, planned path, and statistics.
    Matches the visual style of Activity 1 so students can directly compare.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, (ax_grid, ax_stats) = plt.subplots(
        1, 2, figsize=(13, 6),
        gridspec_kw={'width_ratios': [3, 1]})

    rows, cols = grid.shape
    cs = CELL_SIZE

    ax_grid.set_facecolor('#1A1A2E')
    ax_grid.set_xlim(-0.3, cols * cs + 0.3)
    ax_grid.set_ylim(-0.3, rows * cs + 0.3)
    ax_grid.set_aspect('equal')
    ax_grid.grid(True, linestyle='--', linewidth=0.3, alpha=0.25,
                  color='white')

    # Explored cells
    for (r, c) in explored:
        if grid[r, c] == 0:
            ax_grid.add_patch(plt.Rectangle(
                (c * cs + 0.05, r * cs + 0.05), cs * 0.9, cs * 0.9,
                facecolor='#2980B9', edgecolor='none', alpha=0.30))

    # Shelf obstacles
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == 1:
                ax_grid.add_patch(plt.Rectangle(
                    (c * cs, r * cs), cs, cs,
                    facecolor='#8B6914', edgecolor='#5D4309',
                    linewidth=0.8, alpha=0.95))

    # Path line
    wx = [pt[0] for pt in waypoints]
    wy = [pt[1] for pt in waypoints]
    ax_grid.plot(wx, wy, '-', color='#FF4136', linewidth=2.8,
                  zorder=5, label=f'A* path ({len(waypoints)} waypoints)')

    # Waypoint dots (colour gradient)
    n = len(waypoints)
    for i, wp in enumerate(waypoints):
        t   = i / max(n - 1, 1)
        col = [t, 1.0 - t * 0.7, 1.0 - t]
        ax_grid.scatter(wp[0], wp[1], c=[col], s=30, zorder=6, alpha=0.9)

    # Start / Goal
    sx, sy = cell_to_world(*start_cell)
    gx, gy = cell_to_world(*goal_cell)
    ax_grid.scatter(sx, sy, c='#2ECC71', s=220, zorder=7,
                     edgecolors='white', linewidths=1.2, label='Start')
    ax_grid.scatter(gx, gy, c='#2ECC71', s=320, marker='*', zorder=7,
                     edgecolors='white', linewidths=1.2, label='Goal')

    ax_grid.set_title("Activity 3 — A* Path in Warehouse\n"
                       "(Top-Down View, matches PyBullet scene)",
                       fontsize=12, fontweight='bold', color='white')
    ax_grid.set_xlabel("x (m)", color='white', fontsize=9)
    ax_grid.set_ylabel("y (m)", color='white', fontsize=9)
    ax_grid.tick_params(colors='white')

    legend_handles = [
        mpatches.Patch(facecolor='#2980B9', alpha=0.4,
                       label=f'A* explored ({len(explored)} cells)'),
        plt.Line2D([0], [0], color='#FF4136', linewidth=2.5,
                   label=f'Planned path ({len(waypoints)} waypoints)'),
        mpatches.Patch(facecolor='#8B6914',
                       label='Shelving (C_obs)'),
        mpatches.Patch(facecolor='#1A1A2E',
                       label='Aisles (C_free)'),
    ]
    ax_grid.legend(handles=legend_handles, fontsize=8, loc='upper left',
                    framealpha=0.75, labelcolor='white',
                    facecolor='#2C3E50')

    # Stats panel
    ax_stats.axis('off')
    ax_stats.set_facecolor('#F8F9FA')

    efficiency = (1 - len(explored) / grid.size) * 100

    path_length_cells = len(waypoints) - 1
    path_length_m     = path_length_cells * CELL_SIZE

    stats = [
        ("ALGORITHM",     "A* Search\nf(n) = g(n) + h(n)"),
        ("HEURISTIC",     "Manhattan distance\n(admissible)"),
        ("GRID SIZE",     f"{rows} x {cols} = {grid.size} cells"),
        ("C_OBS CELLS",   f"{int(grid.sum())} (shelves)"),
        ("EXPLORED",      f"{len(explored)} cells\n({100-efficiency:.1f}% of grid)"),
        ("CELLS SKIPPED", f"{grid.size - len(explored)}\n({efficiency:.1f}% efficiency)"),
        ("PATH LENGTH",   f"{path_length_cells} steps\n({path_length_m:.1f} m)"),
        ("WAYPOINTS",     f"{len(waypoints)}"),
        ("NEXT LAB",      "Path Tracking &\nControl (Lab 6)"),
    ]

    y = 0.97
    for label, value in stats:
        ax_stats.text(0.06, y, label, transform=ax_stats.transAxes,
                       fontsize=8, fontweight='bold', color='#555', va='top')
        y -= 0.05
        ax_stats.text(0.06, y, value, transform=ax_stats.transAxes,
                       fontsize=10, color='#1A252F', va='top')
        y -= 0.09

    ax_stats.set_title("Statistics", fontsize=11, fontweight='bold')
    fig.patch.set_facecolor('#1A1A2E')

    plt.tight_layout()
    plt.savefig("activity3_output.png", dpi=130, bbox_inches='tight')
    print("[VIZ] Summary plot saved --> activity3_output.png")
    plt.show()


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    print("=" * 65)
    print("  Activity 3 — Warehouse Path Planning (PyBullet)")
    print("  Planner  : A* (imported from activity1_astar.py)")
    print("  Heuristic: Manhattan distance (admissible)")
    print("  Simulator: PyBullet 3-D warehouse")
    print()
    print("  Focus: PATH PLANNING only.")
    print("  Path Tracking & Control --> Lab 6")
    print("=" * 65)

    # ── Build occupancy grid ──────────────────────────────────────────────
    grid       = build_warehouse_grid()
    start_cell = (1,  1)
    goal_cell  = (10, 10)

    sx, sy = cell_to_world(*start_cell)
    gx, gy = cell_to_world(*goal_cell)

    print(f"\n  Grid     : {GRID_ROWS}x{GRID_COLS} cells  "
          f"({GRID_ROWS * CELL_SIZE:.0f}m x {GRID_COLS * CELL_SIZE:.0f}m)")
    print(f"  Start    : cell {start_cell}  ->  world ({sx:.1f}, {sy:.1f})")
    print(f"  Goal     : cell {goal_cell}  ->  world ({gx:.1f}, {gy:.1f})")
    print()

    # ── Run A* ───────────────────────────────────────────────────────────
    waypoints, explored = run_global_planner(grid, start_cell, goal_cell)
    if waypoints is None:
        print("Cannot visualize — no path found.")
        return

    # ── PyBullet scene ────────────────────────────────────────────────────
    print("\n[SIM] Launching PyBullet ...")
    setup_pybullet()
    load_ground_and_walls()
    load_warehouse_obstacles(grid)

    robot_id = load_robot(start_cell)

    # ── Visualize the planned path ─────────────────────────────────────────
    print("[SIM] Drawing A* path in 3-D scene ...")
    visualize_path_in_pybullet(waypoints, explored, grid)

    # ── Step physics once to settle visuals ───────────────────────────────
    for _ in range(60):
        p.stepSimulation()
        time.sleep(1.0 / 60.0)

    print()
    print("=" * 65)
    print("  What you are seeing in PyBullet:")
    print("  - Blue floor tiles  : cells A* explored during search")
    print("  - Red line          : the planned path (sequence of waypoints)")
    print("  - Coloured spheres  : individual waypoints (cyan -> orange)")
    print("  - Green star        : goal position")
    print("  - Husky robot       : placed at start, stationary (Lab 6 topic)")
    print()
    print("  REFLECTION QUESTIONS (answer in your lab report):")
    print("  Q1. A* explored only a fraction of the grid. Why does it")
    print("      skip so many cells? What role does h(n) play?")
    print("  Q2. The path goes around the shelves even though a straight")
    print("      line from start to goal would be shorter. Why?")
    print("  Q3. Why is path planning (this lab) a separate step from")
    print("      path tracking and control (next lab)?")
    print("=" * 65)
    print()
    print("[SIM] Window open -- press Ctrl+C or close the window to exit.")

    # Keep the window open until the user closes it
    try:
        while p.isConnected():
            p.stepSimulation()
            time.sleep(1.0 / 60.0)
    except Exception:
        pass

    p.disconnect()
    print("[SIM] PyBullet closed.")

    # ── Save matplotlib summary ───────────────────────────────────────────
    print("[VIZ] Generating summary plot ...")
    save_summary_plot(grid, waypoints, explored, start_cell, goal_cell)


if __name__ == "__main__":
    main()


---

## Summary & Next Steps

Congratulations on completing this lab! Before moving on:

1. **Review** your outputs and make sure they match expected behavior
2. **Experiment** by modifying parameters and observing the effects
3. **Reflect** on what you learned — write a brief paragraph in your report

---

*Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/) · Al-Zaytoonah University of Jordan*
